# 04 — Locomotor resilience

Quantifies the impact of the sand perturbation by comparing **inside-the-sand-bed** metrics with **other corridor passages** (baseline).

## Method (hybrid bbox + PCA)

1. Compute 3 resilience signals from the vertical sacrum trace:
   - `sig_var`     — rolling variance (1 s window) → vertical variability
   - `sig_radius`  — phase-space radius (distance to TDE centroid) → dynamic dispersion
   - `sig_cadence` — local dominant frequency (Welch) → instantaneous cadence
2. **Perturbation** = longest bbox segment (precise, median error ≈ 0 s vs CRF)
3. **Baseline** = other corridor passages detected via PCA-calibrated corridor (statistically valid: same circuit phase as perturbation)
4. Per-trial metrics: `z_var`, `z_radius`, `ratio_var`, `ratio_radius`, `delta_cadence_hz`

Exploitability criterion: at least 3 valid baseline passages, otherwise the trial is recorded with NaN metrics.

## Inputs / Outputs

- **Inputs**  : `data/processed/*.npz`, `outputs/tables/03_tde_per_participant.csv`
- **Outputs** : `outputs/tables/04_resilience_metrics.csv`, `outputs/figures/04_metrics_by_condition.png`, `outputs/figures/04_individual_trajectories.png`

In [ ]:
from resilience import paths
from resilience.analysis import resilience_workflow
from resilience.viz import plots

from pathlib import Path
import matplotlib.pyplot as plt
%matplotlib inline

OUT_FIGURES = Path(paths.OUTPUTS_DIR) / 'figures'

## 1. Run pipeline

In [ ]:
tde_params = resilience_workflow.load_tde_params()
pca_calibs = resilience_workflow.calibrate_all_participants()
trials     = resilience_workflow.load_all_trials(pca_calibs)

print(f"\n✅ {len(trials)} trials loaded")

In [ ]:
df_res, signals_cache = resilience_workflow.compute_metrics(trials, tde_params)
print(f"\n✅ {len(df_res)} trials analysed")
df_res

## 2. Descriptive stats per condition

In [ ]:
metric_cols = ['z_var', 'z_radius', 'ratio_var', 'ratio_radius', 'delta_cadence_hz']
for col in metric_cols:
    print(f"\n--- {col} ---")
    print(df_res.groupby('condition')[col].describe()[['count', '25%', '50%', '75%']].round(2))

## 3. Plots

In [ ]:
plots.plot_resilience_boxplots(df_res,
    save_to=OUT_FIGURES / '04_metrics_by_condition.png')
plt.show()

In [ ]:
plots.plot_resilience_individual_trajectories(df_res,
    save_to=OUT_FIGURES / '04_individual_trajectories.png')
plt.show()

## 4. Export

In [ ]:
out_path = resilience_workflow.export(df_res)
n_complete = df_res[metric_cols].notna().all(axis=1).sum()
print(f"✅ Exported: {out_path}")
print(f"   {len(df_res)} trials, {df_res['participant'].nunique()} participants")
print(f"   {n_complete} trials with all 5 metrics valid")